# P1-SHAPE frozen backbone smoke

Runs one series for each of four untouched mechanisms and three frozen seeds. This is smoke evidence only and cannot compute the P1 decision. Select a T4 GPU and run once for each backbone.

In [ ]:
BACKBONE = 'chronos_2'  # change to 'timesfm_3' for the second smoke
assert BACKBONE in {'chronos_2', 'timesfm_3'}
print('Selected frozen backbone:', BACKBONE)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess

REPO = Path('/content/tsfm-covariate-faithfulness')
REPO_URL = 'https://github.com/FlyMe2star/tsfm-covariate-faithfulness.git'
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'main', REPO_URL, str(REPO)], check=True)
print('Repository commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
import os
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO / 'requirements/colab-base.txt')], check=True)
model_requirements = REPO / ('requirements/chronos2.txt' if BACKBONE == 'chronos_2' else 'requirements/timesfm3.txt')
if BACKBONE == 'timesfm_3':
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'timesfm'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(model_requirements)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
src_path = str(REPO / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print('Dependencies installed for', BACKBONE)

In [ ]:
import hashlib
import json
import torch

from covfaith.config import load_yaml, verify_config_lock

assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU, then restart.'
config_path = REPO / 'configs/p1_shape/covintervene_shape_p1.yaml'
lock_path = REPO / 'configs/p1_shape/covintervene_shape_p1.lock.json'
config_hash = verify_config_lock(config_path, lock_path)
construct_path = REPO / 'evidence/p1_shape/construct/metric_construct_validation.yaml'
receipt = json.loads((construct_path.parent / 'metric_construct_validation.receipt.json').read_text())
assert hashlib.sha256(construct_path.read_bytes()).hexdigest() == receipt['sha256']
assert receipt['all_checks_passed'] is True
test_env = os.environ.copy()
test_env['PYTHONPATH'] = src_path
subprocess.run([sys.executable, '-m', 'pytest', '-q', str(REPO / 'tests')], cwd=REPO, env=test_env, check=True)
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)
print('Frozen P1-SHAPE config hash:', config_hash[:12])

In [ ]:
from covfaith.adapters import Chronos2Adapter, TimesFM3Adapter

config = load_yaml(config_path)
model = next(item for item in config['models'] if item['id'] == BACKBONE)
if BACKBONE == 'chronos_2':
    adapter = Chronos2Adapter.from_pretrained(model['checkpoint'], model['revision'], device='cuda', batch_size=128)
else:
    adapter = TimesFM3Adapter.from_pretrained(model['checkpoint'], model['revision'], device='cuda', per_core_batch_size=16)
print('Frozen checkpoint loaded:', model['checkpoint'], model['revision'])

In [ ]:
from covfaith.p1_shape import run_shape_backbone_units

OUTPUT_ROOT = Path('/content/drive/MyDrive/tsfm-covariate-faithfulness/p1_shape_smoke_v1')
report = run_shape_backbone_units(REPO, adapter, OUTPUT_ROOT, smoke_count=1)
assert report['scientific_gate_computed'] is False
assert report['completed_unit_count'] == 12
print(json.dumps(report, indent=2, ensure_ascii=False))
print('Smoke artifacts:', OUTPUT_ROOT)

Send the final JSON report to Codex. Do not run the full matrix until both backbone smoke reports pass.